## Limpieza básica “universal” (interacciones)

### Objetivos

1. Asegurar tipos y rangos: user_id, recipe_id como int; rating válido
2. Quitar filas problemáticas: NaN, rating fuera de [0,5] (o el rango que tenga)
3. Quitar duplicados (si los hay): mismas (user, recipe, date) o (user, recipe)
4. Hacer un filtrado suave de sparsity que ayuda a todos los modelos:
    * usuarios con ≥ min_user_interactions
    * recetas con ≥ min_item_interactions
5. Asegurar consistencia entre splits: que no haya usuarios/items en val/test que no existan en train (si aparecen, se eliminan o se “mapean a unknown”; lo más estándar es eliminarlos para evaluación justa)

In [4]:
import os
from pathlib import Path

print("CWD:", os.getcwd())
print("Existe data/raw?:", Path("data/raw").exists())
print("Existe el archivo?:", Path("data/raw/interactions_train.csv").exists())

CWD: /Users/diegorecover/Desktop/Diego - CDIA/Cursos/Segundo/Segundo Semestre/Computacion Social/Bloque 2. Personalización/Food.com-Recommender-Systems/notebooks
Existe data/raw?: False
Existe el archivo?: False


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data/raw")  # aquí tienes los csv locales

train = pd.read_csv(DATA_DIR / "interactions_train.csv")
val   = pd.read_csv(DATA_DIR / "interactions_validation.csv")
test  = pd.read_csv(DATA_DIR / "interactions_test.csv")

print(train.shape, val.shape, test.shape)
train.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/interactions_train.csv'

In [ ]:
def basic_clean_interactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- columnas esperadas ---
    required = {"user_id", "recipe_id", "rating"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas obligatorias: {missing}")

    # --- tipos ---
    df["user_id"]  = pd.to_numeric(df["user_id"], errors="coerce")
    df["recipe_id"] = pd.to_numeric(df["recipe_id"], errors="coerce")
    df["rating"]   = pd.to_numeric(df["rating"], errors="coerce")

    # --- eliminar NaNs en claves/rating ---
    df = df.dropna(subset=["user_id", "recipe_id", "rating"])

    df["user_id"] = df["user_id"].astype(int)
    df["recipe_id"] = df["recipe_id"].astype(int)

    # --- rating razonable (Food.com suele ser 0-5 o 1-5; aquí dejamos 0-5) ---
    df = df[(df["rating"] >= 0) & (df["rating"] <= 5)]

    # --- fecha opcional ---
    if "date" in df.columns:
        # convertir sin romper si hay formatos raros
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # --- eliminar duplicados (criterio universal) ---
    # Si hay date, usamos (user, item, date). Si no, (user, item).
    subset = ["user_id", "recipe_id", "date"] if "date" in df.columns else ["user_id", "recipe_id"]
    df = df.drop_duplicates(subset=subset, keep="last")

    return df

train_c = basic_clean_interactions(train)
val_c   = basic_clean_interactions(val)
test_c  = basic_clean_interactions(test)

print("Después de limpieza:")
print(train_c.shape, val_c.shape, test_c.shape)

In [ ]:
def enforce_train_only_entities(train_df, val_df, test_df):
    """Elimina de val/test usuarios o items que no aparezcan en train."""
    train_users = set(train_df["user_id"].unique())
    train_items = set(train_df["recipe_id"].unique())

    def filter_df(df):
        return df[df["user_id"].isin(train_users) & df["recipe_id"].isin(train_items)].copy()

    val_f  = filter_df(val_df)
    test_f = filter_df(test_df)

    return val_f, test_f

val_c, test_c = enforce_train_only_entities(train_c, val_c, test_c)

print("Tras restringir val/test a entidades vistas en train:")
print(train_c.shape, val_c.shape, test_c.shape)

In [ ]:
def apply_min_interactions(train_df, val_df, test_df, min_user=5, min_item=5):
    """
    Filtra usuarios e items con pocas interacciones.
    Lo calculamos SOLO en train (para no 'mirar' test).
    Luego aplicamos el filtro a train/val/test.
    """
    user_counts = train_df["user_id"].value_counts()
    item_counts = train_df["recipe_id"].value_counts()

    keep_users = set(user_counts[user_counts >= min_user].index)
    keep_items = set(item_counts[item_counts >= min_item].index)

    def filter_df(df):
        return df[df["user_id"].isin(keep_users) & df["recipe_id"].isin(keep_items)].copy()

    return filter_df(train_df), filter_df(val_df), filter_df(test_df)

MIN_USER = 5
MIN_ITEM = 5

train_f, val_f, test_f = apply_min_interactions(train_c, val_c, test_c, min_user=MIN_USER, min_item=MIN_ITEM)

print("Tras filtro mínimo de interacciones:")
print(train_f.shape, val_f.shape, test_f.shape)
print("Usuarios:", train_f["user_id"].nunique(), "Items:", train_f["recipe_id"].nunique())

In [ ]:
def apply_min_interactions(train_df, val_df, test_df, min_user=5, min_item=5):
    """
    Filtra usuarios e items con pocas interacciones.
    Lo calculamos SOLO en train (para no 'mirar' test).
    Luego aplicamos el filtro a train/val/test.
    """
    user_counts = train_df["user_id"].value_counts()
    item_counts = train_df["recipe_id"].value_counts()

    keep_users = set(user_counts[user_counts >= min_user].index)
    keep_items = set(item_counts[item_counts >= min_item].index)

    def filter_df(df):
        return df[df["user_id"].isin(keep_users) & df["recipe_id"].isin(keep_items)].copy()

    return filter_df(train_df), filter_df(val_df), filter_df(test_df)

MIN_USER = 5
MIN_ITEM = 5

train_f, val_f, test_f = apply_min_interactions(train_c, val_c, test_c, min_user=MIN_USER, min_item=MIN_ITEM)

print("Tras filtro mínimo de interacciones:")
print(train_f.shape, val_f.shape, test_f.shape)
print("Usuarios:", train_f["user_id"].nunique(), "Items:", train_f["recipe_id"].nunique())

---

In [ ]:
def quick_stats(df, name):
    n_users = df["user_id"].nunique()
    n_items = df["recipe_id"].nunique()
    n_inter = len(df)
    density = n_inter / (n_users * n_items)
    print(f"\n{name}")
    print("Interactions:", n_inter)
    print("Users:", n_users, "Items:", n_items)
    print("Sparsity (density):", density)

quick_stats(train_f, "TRAIN")
quick_stats(val_f, "VAL")
quick_stats(test_f, "TEST")

# Distribución rating
train_f["rating"].value_counts().sort_index()

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
train_f["user_id"].value_counts().hist(bins=50)
plt.yscale("log")
plt.title("Interacciones por usuario (log)")
plt.xlabel("#interacciones"); plt.ylabel("frecuencia (log)")
plt.show()

plt.figure()
train_f["recipe_id"].value_counts().hist(bins=50)
plt.yscale("log")
plt.title("Interacciones por receta (log)")
plt.xlabel("#interacciones"); plt.ylabel("frecuencia (log)")
plt.show()